# Goals:

1) get RSA working in python

2) decide how to distribute (Google CoLab, make a website, etc.)

3) make it pretty

4) make it fast

5) make it understandable

In [1]:
p = 1
q = 1
n = p * q
e = 65537  # maybe change this to make things easier?

# lamb = None 
# d = None

So p and q we can set manually for now.  n is pretty easy its just multiplication.  e has a standard value (although I think I'd like to make it smaller for ease of demonstration).  So I just need a way of calculating lambda(n) and d.  Let's start with Carmichael's Totient.

Even on the wikipedia article, the explain that this is an advanced version of euclid's algorithm.  I bet there's a builtin math function for lcm (whoever wrote the code for the paper has used `math.gcd()` somewhere.  I don't think its unreasonable to use `math.lcm()` then).  

Carmichael's totient is a very interesting function.  examine $\lambda(p)$ for $p$ prime.  we know $\phi(p)=p-1$ since every number less than a prime will be coprime to it.  Furthermore we know $\mathbb{Z}_p^\times\equiv\left(\mathbb{Z}/p\mathbb{Z}\right)^{\times}$ is cyclic with order $p-1$.  Then exactly one element of $\mathbb{Z}_p^\times$ has order $p-1$.  Then $\lambda(p)=p-1$

But again we are not working modulo a prime in RSA.  Instead we work modulo a semiprime.  So we must use the fact $\lambda(pq)=\text{lcm}\left[\lambda(p), \lambda(q)\right]$, i.e. $\lambda(pq)=\text{lcm}\left[p-1,\,q-1\right]$.  Which to be fair is exactly what it says in the wikipedia article... but I had to think through the proof myself I guess...

anyway lets check if theres an lcm builtin in the `math` module.

Yes! `math.lcm()` exists.  We'll also import `math.gcd()` while we're at it.  

In [2]:
from math import lcm, gcd

In [3]:
def carmichael(p: int, q: int) -> int:
    """Return Carmichael's totient of p*q, only if p and q are prime.
    
    This function returns the least common multiple of p-1 and q-1. 
    If p and q are prime, this is Carmichael's totient of their 
    product.  This is helpful for the RSA cryptosystem.  
    """
    return lcm(p-1, q-1)

Good so now we can find $\lambda(pq)$ very easily.  The next step is to calculate the inverse of some number modulo $\lambda(n)$.

As the wikipedia article mentions, since $e$ and $\lambda(n)$ are coprime (because $e$ is prime) we can use the extended euclidean algorithm to find such a number. 

I'm tempted to write the extended euclidean algorithm out myself, because that sounds fun.  Let's see how difficult that is.

So the idea is that $d\cdot e+x\cdot\lambda(n)=1$ for some $d$ and some $x$ thanks to Bezout's identity.  Then $d$ is the multiplicative inverse of $e$ modulo $\lambda(n)$

Okay the wikipedia version of this algorithm is not hard to implement. This is very straightforward in python.  

In [4]:
def bezout(a: int, b: int) -> tuple:
    """Return a tuple containing gcd(a, b), and the Bezout coefficients of a and b."""
    s0 = 1
    t0 = 0
    s1 = 0
    t1 = 1
    if a==b:
        return abs(a), 1, 0
    elif abs(a)>abs(b):
        r0 = abs(a)
        r1 = abs(b)
    elif abs(a)<abs(b):
        r0 = abs(b)
        r1 = abs(a)
    else:
        raise RuntimeError("What the fuck??")

    while not r1==0:
        q1 = r0 // r1
        r2 = r0 % r1
        s2 = s0 - (q1*s1)
        t2 = t0 - (q1*t1)
        r0 = r1
        r1 = r2
        s0 = s1
        s1 = s2
        t0 = t1
        t1 = t2
    if abs(a)>abs(b):
        return r0, s0, t0
    if abs(a)<abs(b):
        return r0, t0, s0


In [5]:
bezout(240, 46)

(2, -9, 47)

In [6]:
(-9*240)+(47*46)

2

based.

In [7]:
def modinverse(a: int, n: int) -> int:
    """Return the multiplicative inverse of a modulo n."""
    x, y, z = bezout(a, n)
    if y>0:
        return y
    elif y<0:
        return y + int(n/x)
    else:
        raise RuntimeError("This should never happen.")

In [8]:
modinverse(2, 5)

3

So in theory, now we can fully implement RSA, given some primes p and q.

In [9]:
p = 7
q = 11
n = p*q
lamb = carmichael(p, q)
print(lamb)

30


In [10]:
e = 17
d = modinverse(e, lamb)
print(d)

23


In [11]:
d*e

391

In [12]:
391%30

1

So we might have:

modulus $n=77$

public key $e=17$

private key $d=23$

In [13]:
for x in range(20):
    print(x, (x**e)%n, (x**(e*d))%n, sep="    ")

0    0    0
1    1    1
2    18    2
3    75    3
4    16    4
5    3    5
6    41    6
7    28    7
8    57    8
9    4    9
10    54    10
11    44    11
12    45    12
13    62    13
14    42    14
15    71    15
16    25    16
17    19    17
18    72    18
19    24    19


Fuck yeah RSA cryptosystem (basically)